<img src="https://raw.githubusercontent.com/brazil-data-cube/code-gallery/master/img/logo-bdc.png" align="right" width="64" style="margin-top: 10px;" />

<h1 style="color:#336699; font-weight: normal; margin-top: 0;">Classificação de Áreas Agrícolas no Oeste da Bahia</h1>
<hr style="border: 2px solid #0077b9; margin-top: 5px; margin-bottom: 20px;" />

<br/>

<div style="text-align: center; font-size: 90%; color: #333;">
    Felipe Carlos<sup><a href="https://orcid.org/0000-0002-3334-4315" target="_blank">
    <i class="fab fa-lg fa-orcid" style="color: #a6ce39;"></i></a></sup>, 
    Felipe Souza<sup><a href="https://orcid.org/0000-0002-5826-1700" target="_blank">
    <i class="fab fa-lg fa-orcid" style="color: #a6ce39;"></i></a></sup>, 
    Gilberto Camara<sup><a href="https://orcid.org/0000-0002-3681-487X" target="_blank">
        <i class="fab fa-lg fa-orcid" style="color: #a6ce39;"></i></a></sup>, 
    Rolf Simoes<sup><a href="https://orcid.org/0000-0003-0953-4132" target="_blank">
        <i class="fab fa-lg fa-orcid" style="color: #a6ce39;"></i></a></sup>, 
    <br/><br/>
    Divisão de Geoinformática e Observação da Terra, Instituto Nacional de Pesquisas Espaciais (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brazil
    <br/><br/>
    Contatos: <a href="mailto:efelipecarlos@gmail.com">efelipecarlos@gmail.com</a>, <a href="mailto:lipecaso@gmail.com">lipecaso@gmail.com</a>
</div>

<br/>

<div style="text-align: justify; margin-left: 15%; margin-right: 15%;">
    <b>Resumo</b><br>
    <code>sits</code> é um software de código aberto voltado a análise de séries temporais de imagens de satélite. Este notebook apresenta as principais funcionalidades do <code>sits</code>, partindo da criação de um cubo de dados até a geração de um mapa de classificação de uso e cobertura da terra. 

</div>    

<br/>

<div style="text-align: justify; margin-left: 15%; margin-right: 15%; font-size: 90%; border: 1px solid #0077b9; border-radius: 6px; padding: 10px;">
    <b>Para um aprofundamento e  discussão completa sobre o <code>sits</code>, consulte o livro em: </b><a href="https://e-sensing.github.io/sitsbook/" target="_blank">SITS: Satellite Image Time Series Analysis on Earth Observation Data Cubes</a>.
</div>


## Ambiente
<hr style="border:1px solid #0077b9;">

Esta seção configura o ambiente de execução deste notebook. As configurações apresentadas não são requisitos do `sits`, mas sim do ambiente utilizado no minicurso.

Instalação do `sits` 2.0.0, a versão descrita no livro:

In [ ]:
devtools::install_cran("kohonen")
devtools::install_github("e-sensing/sits@dev", dependencies = FALSE)

Configuração das dimensões dos plots:

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 12)

## Workflow
<hr style="border:1px solid #0077b9;">

O pacote `sits` disponibiliza funções simples que possibilitam uma rápida criação de mapas de uso e cobertura da terra. **Apenas 6 funções** são necessárias para fazer uma classificação. Essas funções são apresentadas abaixo: 

<div style="text-align: center">
    <video src="videos/00-workflow.mp4" controls preload="metadata" width="85%"></video>
</div>

Além dessas funções, diversas outras estão disponíveis no `sits`, servindo como funcionalidades auxiliares ao fluxo principal apresentado acima.

## Inicialização
<hr style="border:1px solid #0077b9;">

Para começar o uso do `sits`, execute o seguinte comando:

In [ ]:
library(sits)

## Cubos de dados com sits
<hr style="border:1px solid #0077b9;">

Atualmente, há uma grande variedade de dados de Observação da Terra disponíveis por meio de diferentes provedores. O pacote `sits` oferece funções simples que permitem criar cubos de dados a partir dessas fontes, incluindo **B**razil **D**ata **C**ube (BDC), **A**mazon **W**eb **S**ervices (AWS), **M**icrosoft **P**lanetary **C**omputer (MPC), entre outros.

A animação abaixo ilustra como a função `sits_cube()` pode ser utilizada para criar um cubo de dados a partir de diferentes provedores:

<div style="text-align: center">
    <video src="videos/01-sits-cube.mp4" controls preload="metadata" width="80%"></video>
</div>

### Carregando cubos de dados do Brazil Data Cube (INPE)

Seguindo o exemplo apresentado na animação acima, podemos carregar um cubo com o seguinte comando:

In [ ]:
cube <- sits_cube(
    source     = "BDC",
    collection = "SENTINEL-2-16D",
    tiles      = "030019",
    start_date = "2019-09-30",
    end_date   = "2020-09-29",
    bands      = c("B02", "B03", "B04", "B8A", "B11", "NDVI", "CLOUD")
)

In [ ]:
print(cube)

Ao criar o cubo, o `sits` não baixa nenhuma imagem. Ele envia a descrição do cubo ao catálogo STAC do provedor, que responde apenas com **metadados**: datas, bandas e os links para os arquivos. Os dados continuam no provedor e só são lidos quando uma etapa seguinte precisa deles, como a visualização, a extração de séries temporais ou a classificação. Por isso, as funções abaixo respondem rapidamente: elas apenas consultam esses metadados.

<div style="text-align: center">
    <video src="videos/01b-no-pixels.mp4" controls preload="metadata" width="85%"></video>
</div>

<details style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; padding: 10px 15px; background-color: #fdfdfd; margin-top: 10px; box-shadow: 0 1px 3px rgba(0,0,0,0.05); color: #333;">
  <summary style="font-weight: bold; cursor: pointer; outline: none;"> 🔍 Mais opções para filtragem de dados</summary>

  ### Filtro temporal

  Caso queira carregar um cubo de dados com imagens de um período temporal diferente, é possível alterar os parâmetros `start_date` e `end_date` da função `sits_cube()`. Abaixo, são apresentados alguns exemplos:

  #### Cubo com dados de 2018

  ```R
    cube <- sits_cube(
        # Provedor
        source = "BDC",
    
        # Coleção
        collection = "SENTINEL-2-16D",
        
        # Extensão espacial
        tiles = "030019",
    
        # Extensão temporal
        start_date = "2018-01-01",
        end_date = "2018-09-30"
    )
  ```

  #### Cubo com dados de 2022

  ```R
    cube <- sits_cube(
        # Provedor
        source = "BDC",
    
        # Coleção
        collection = "SENTINEL-2-16D",
        
        # Extensão espacial
        tiles = "030019",
    
        # Extensão temporal
        start_date = "2022-01-01",
        end_date = "2022-09-30"
    )
  ```

  ### Filtro espacial

  O `sits` também disponibiliza multiplas formas de selectionar a região de interesse. No exemplo acima, foi mostrado o uso do argumento `tiles`, que especifica o tile BDC desejado. Além desse argumento, também é possível utilizar o `roi`. Esse parâmetro permite especificar a região de interesse por meio um shapefile, bbox ou outros formatos.

  No `sits_cube()`, o `roi` apenas seleciona os tiles que intersectam a região, ele **não** recorta as imagens. Para recortar, use o `roi` no `sits_classify()` ou `sits_cube_copy()`.

  Abaixo, são apresentados exemplos de como esse parâmetro pode ser utilizado:

  #### Utilização do ROI com shapefile

  ```R
    cube <- sits_cube(
        # Provedor
        source = "BDC",
    
        # Coleção
        collection = "SENTINEL-2-16D",
        
        # Extensão espacial
        roi = "/caminho/até/o/shapefile",
    
        # Extensão temporal
        start_date = "2022-01-01",
        end_date = "2022-09-30"
    )
  ```

  #### Utilização do ROI com objeto sf

  ```R
    sf_object <- sf::read_sf("/caminho/até/o/gpkg")


    cube <- sits_cube(
        # Provedor
        source = "BDC",
    
        # Coleção
        collection = "SENTINEL-2-16D",
        
        # Extensão espacial
        roi = sf_object,
    
        # Extensão temporal
        start_date = "2022-01-01",
        end_date = "2022-09-30"
    )
  ```
</details>

<div style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; background-color: #fdfdfd; padding: 15px 20px; font-size: 15px; color: #333; line-height: 1.5; box-shadow: 0 1px 3px rgba(0, 0, 0, 0.05);">
  Para mais informações sobre os provedores de dados suportados pelo <code>sits</code>, consulte o capítulo 
  <a href="https://e-sensing.github.io/sitsbook/dc_ardcollections.html" target="_blank" style="color: #1f70b8; text-decoration: none;">
    Analysis-ready image collections
  </a> do livro oficial do <code>sits</code>.
</div>

### Explorando características do cubo 
<hr style="border:1px solid #0077b9;">

In [ ]:
sits_timeline(cube)

In [ ]:
sits_bands(cube)

<details style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; padding: 10px 15px; background-color: #fdfdfd; box-shadow: 0 1px 3px rgba(0,0,0,0.05); color: #333;">
  <summary style="font-weight: bold; cursor: pointer; outline: none;"> 🔍 Mais sobre multiplas bandas em cubos</summary>

  <p style="margin-top: 5px">
      Ao criar um cubo, o <code>sits</code> carrega automaticamente as bandas e datas selecionadas. A animação abaixo ilustra como um cubo composto de várias bandas é materializado.
  </p>

  <div style="text-align: center">
    <video src="videos/02-cube-structure.mp4" controls preload="metadata" width="80%"></video>
 </div>
</details>

### Visualizando o cubo
<hr style="border:1px solid #0077b9;">

In [ ]:
# Plot the cube
plot(cube, dates = "2019-09-30", red = "B11", green = "B8A", blue = "B02")

<details style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; padding: 10px 15px; margin: 20px 0; background-color: #fdfdfd; box-shadow: 0 1px 3px rgba(0,0,0,0.05); color: #333;">
  <summary style="font-weight: bold; cursor: pointer; outline: none;"> 📊 Mais opções de visualização</summary>

  ### Mais detalhes no `plot()`
  O `sits` permite adicionar mais informações no plot, como especificar uma data com o argumento `dates` e selecionar bandas específicas com o argumento `band`:
  
  ```R
  plot(cube, band = "NDVI", dates = "2020-07-11")
  ```
</details>

## Extração de séries temporais do cubo
<hr style="border:1px solid #0077b9;">

Com o cubo criado, é possível extrair séries temporais associadas a amostras de uso e cobertura da terra. Essa operação permite associar o comportamento espectro-temporal às amostras rotuladas.

A animação abaixo ilustra como esse processo pode ser realizado com a função `sits_get_data()`:

<div style="text-align: center">
    <video src="videos/03-sits-get-data.mp4" controls preload="metadata" width="80%"></video>
</div>

### Extraíndo séries temporais de amostras no oeste da Bahia
<hr style="border:1px solid #0077b9;">

Carrega as amostras:

In [ ]:
samples <- sf::st_read("https://github.com/e-sensing/sitsnotebooks/raw/refs/heads/main/GEOPANTANAL_2026/data/raw/samples/LEM_samples.gpkg")

In [ ]:
print(head(samples))

Vamos carregar também as cores associadas a cada uma dos rótulos.

In [ ]:
color_table <- readRDS(url("https://github.com/e-sensing/sitsnotebooks/raw/refs/heads/main/GEOPANTANAL_2026/data/raw/colors/colors_lem.rds"))

A função `sits_colors_set` é utilizada para indicar para o `sits` quais cores devem ser utilizadas para cada uma dos rótulos disponíveis no conjunto de amostras.

In [ ]:
sits_colors_set(colors = color_table, legend = "LEM")

Visualiza as amostras:

In [ ]:
sits_view(samples)

Extrai as séries temporais:

In [ ]:
samples_ts <- sits_get_data(
    cube       = cube,
    samples    = samples,
    multicores = 8
)

# Em caso de problemas de conexão, utilize o seguinte comando:
# samples_ts <- readRDS(url("https://github.com/e-sensing/sitsnotebooks/raw/refs/heads/main/GEOPANTANAL_2026/data/derived/samples/LEM-samples-ts.rds"))

<details style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; padding: 10px 15px; background-color: #fdfdfd; box-shadow: 0 1px 3px rgba(0,0,0,0.05); color: #333;">
  <summary style="font-weight: bold; cursor: pointer; outline: none;"> 📈 Mais opções de extração</summary>

  ### Extração de séries temporais com polígonos
  A função `sits_get_data()` suporta o uso de pontos e polígonos. As amostras deste notebook são pontos. Os exemplos abaixo valem para amostras em polígonos. Para polígonos, é possível definir a quantidade de pontos que serão extraídos de cada polígono usando o parâmetro `n_sam_pol`.

    
  ```R
  samples_ts <- sits_get_data(
    # Cubo de dados que será usado para extração
    cube = cube,

    # Amostras usadas para a extração
    samples = samples,

    # Número de pontos que serão extraídos em cada polígonos
    n_sam_pol = 10,

    # Número de processos usados para otimizar a extração 
    multicores = 2
  )
```

  Com `pol_avg = TRUE`, os pontos de cada polígono são resumidos em uma única série (média). O parâmetro `sampling_type` (`"random"`, `"hexagonal"`, `"regular"` ou `"fibonacci"`) controla como os pontos são sorteados.

  ```R
  samples_ts <- sits_get_data(
    # Cubo de dados que será usado para extração
    cube = cube,

    # Amostras usadas para a extração
    samples = samples,

    # Número de pontos que serão extraídos em cada polígonos
    n_sam_pol = 10,

    # Sumariza os pontos de um polígono aplicando a média
    pol_avg = TRUE,

    # Número de processos usados para otimizar a extraação 
    multicores = 2
  )
```
</details>

### Explorando as séries temporais
<hr style="border:1px solid #0077b9;">

Antes de treinar um modelo, vale a pena conhecer as amostras. A qualidade do mapa final depende diretamente da qualidade das amostras: rótulos errados, classes com poucas amostras ou séries muito parecidas entre classes diferentes aparecem depois como erros na classificação.

#### A tabela de séries temporais
<hr style="border:1px solid #0077b9;">

O `sits` armazena as séries temporais em uma tabela (`tibble`). Cada linha é uma amostra, e as colunas guardam:

- `longitude` e `latitude`: localização da amostra;
- `start_date` e `end_date`: período da série temporal;
- `label`: classe atribuída à amostra;
- `cube`: cubo de dados de onde a série foi extraída;
- `time_series`: a série temporal em si, guardada como uma tabela dentro da tabela.

A animação abaixo apresenta o formato da tabela de séries temporais:
<div style="text-align: center; margin-top: 10px">
    <video src="videos/04b-timeseries-structure.mp4" controls preload="metadata" width="80%"></video>
 </div>

In [ ]:
print(samples_ts)

A coluna `time_series` contém, para cada amostra, uma tabela com uma linha por data e uma coluna por banda. Veja a série da primeira amostra:

In [ ]:
samples_ts$time_series[[1]]

#### Atributos das amostras
<hr style="border:1px solid #0077b9;">

Bandas disponíveis nas séries:

In [ ]:
sits_bands(samples_ts)

Datas das séries:

In [ ]:
sits_timeline(samples_ts)

Classes (rótulos) presentes nas amostras:

In [ ]:
sits_labels(samples_ts)

Quantidade e proporção de amostras por classe:

In [ ]:
summary(samples_ts)

<div style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; background-color: #fdfdfd; padding: 15px 20px; margin: 20px 0; font-size: 15px; color: #333; line-height: 1.5; box-shadow: 0 1px 3px rgba(0, 0, 0, 0.05);">
  Observe as colunas <code>count</code> e <code>prop</code>: algumas classes têm centenas de amostras, enquanto outras têm apenas 10. Voltaremos a esse ponto na seção sobre <b>desbalanceamento</b>.
</div>

#### Selecionando e filtrando amostras
<hr style="border:1px solid #0077b9;">

A função `sits_select()` seleciona bandas, classes e datas. Por exemplo, as amostras de algodão (`Cotton`) na banda NDVI:

In [ ]:
cotton_ndvi <- sits_select(samples_ts, bands = "NDVI", labels = "Cotton")

summary(cotton_ndvi)

Como as amostras são uma tabela, também é possível filtrá-las com o `dplyr`. Abaixo, mantemos apenas as amostras de soja (`Soybean`) e algodão (`Cotton`):

In [ ]:
soy_cotton <- dplyr::filter(samples_ts, label %in% c("Soybean", "Cotton"))

summary(soy_cotton)

O mesmo vale para os outros atributos. Por exemplo, as amostras a leste de uma longitude:

In [ ]:
samples_east <- dplyr::filter(samples_ts, longitude > -46)

summary(samples_east)

<details style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; padding: 10px 15px; background-color: #fdfdfd; box-shadow: 0 1px 3px rgba(0,0,0,0.05); color: #333;">
  <summary style="font-weight: bold; cursor: pointer; outline: none;"> 🔍 Mais sobre seleção em séries temporais</summary>

  <p style="margin-top: 5px">
      A função <code>sits_select</code> permite selecionar bandas, classes e datas das séries temporais. Os exemplos abaixo mostram como essa operação pode ser utilizada:
  </p>

  #### Selecionando datas

  ```R
  sits_select(
    samples_ts,
    start_date = "2019-12-03",
    end_date = "2020-01-17"
  )
  ```

  #### Selecionando bandas

  ```R
  sits_select(
      samples_ts,
      bands = c("B8A", "NDVI")
  )
  ```

  #### Selecionando classes

  ```R
  sits_select(
      samples_ts,
      labels = c("Soybean", "Cotton")
  )
  ```

   <br/>

  **Nota**: A função `sits_select` também pode ser utilizada em cubos de dados.
</details>

#### Visualizando séries temporais
<hr style="border:1px solid #0077b9;">

O `plot()` de uma única amostra exibe todas as suas bandas ao longo do tempo:

In [ ]:
plot(samples_ts[1, ])

Ao plotar várias amostras de uma mesma classe e banda, o `sits` sobrepõe todas as séries. Isso mostra a **variabilidade** da classe: quanto mais espalhadas as curvas, mais heterogênea é a classe.

In [ ]:
plot(cotton_ndvi)

Compare com as amostras de soja (`Soybean`):

In [ ]:
plot(sits_select(samples_ts, bands = "NDVI", labels = "Soybean"))

#### Visualizando padrões temporais
<hr style="border:1px solid #0077b9;">

Os padrões temporais resumem cada classe em uma única curva que ressalta os principais comportamentos presentes nas séreis temporais extraídas, facilitando a comparação entre classes. 

A animação abaixo ilustra como esse processo pode ser realizado com a função `sits_patterns()`:

<div style="text-align: center">
    <video src="videos/04c-timeseries-patterns.mp4" controls preload="metadata" width="85%"></video>
</div>

Para visualizar esse comportamento, vamos primeiro selecionar uma banda específica das amostras, neste caso `NDVI`:

In [ ]:
samples_indexes <- sits_select(samples_ts, bands = "NDVI")

Agora, vamos gerar os padrões temporais com o `sits_patterns()`. Nessa operação, o `sits` está gerando um **M**odelo **A**ditivo **G**eneralizado (GAM) para obter uma única série temporal com base em aproximação estatística:

In [ ]:
patterns <- sits_patterns(samples_indexes)

Visualização dos padrões temporais:

In [ ]:
plot(patterns)

<div style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; background-color: #fdfdfd; padding: 15px 20px; margin: 20px 0; font-size: 15px; color: #333; line-height: 1.5; box-shadow: 0 1px 3px rgba(0, 0, 0, 0.05);">
  Classes com padrões muito parecidos, por exemplo, diferentes sucessões de culturas com soja, tendem a ser mais difíceis de separar pelo classificador.
</div>

#### Avaliando a qualidade das amostras com SOM
<hr style="border:1px solid #0077b9;">

Os mapas auto-organizáveis (*Self-Organizing Maps*, SOM) são uma rede neural que organiza as séries temporais em uma grade 2D de **neurônios**. Séries parecidas são agrupadas no mesmo neurônio ou em neurônios vizinhos. Cada neurônio recebe o rótulo da classe mais frequente entre as amostras que ele agrupou.

Com isso, é possível identificar:

- classes que se **misturam** com outras (séries parecidas com rótulos diferentes);
- amostras possivelmente **ruidosas**, cujo rótulo não combina com o do neurônio em que caíram.

<div style="text-align: center">
    <video src="videos/04d-som.mp4" controls preload="metadata" width="85%"></video>
</div>

Gera o mapa SOM usando três bandas das amostras:

In [ ]:
set.seed(123)

som_cluster <- sits_som_map(
    data      = sits_select(samples_ts, bands = c("NDVI", "B8A", "B11")),
    grid_xdim = 15,
    grid_ydim = 15,
    rlen      = 20,
    distance  = "euclidean"
)

<details style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; padding: 10px 15px; background-color: #fdfdfd; box-shadow: 0 1px 3px rgba(0,0,0,0.05); color: #333;">
  <summary style="font-weight: bold; cursor: pointer; outline: none;"> ⚙️ Sobre os parâmetros do SOM</summary>

  - `grid_xdim` e `grid_ydim`: tamanho da grade de neurônios;
  - `rlen`: número de vezes que o conjunto de amostras é apresentado à rede durante o treinamento;
  - `distance`: medida de distância entre as séries. A distância `"dtw"` (*Dynamic Time Warping*) costuma separar melhor as classes, mas é bem mais lenta. Neste notebook usamos `"euclidean"` e três bandas para que o mapa seja gerado em poucos segundos.
</details>

Visualiza o mapa SOM. Cada círculo é um neurônio, colorido pela classe majoritária. Neurônios em cinza não receberam amostras:

In [ ]:
plot(som_cluster)

A função `sits_som_evaluate_cluster()` mede, para cada classe, quanto as suas amostras se misturam com amostras de outras classes:

In [ ]:
som_eval <- sits_som_evaluate_cluster(som_cluster)

plot(som_eval)

A função `sits_som_clean_samples()` classifica cada amostra em três grupos, com base na probabilidade *a priori* (do neurônio) e *a posteriori* (da vizinhança):

- `clean`: amostra consistente com o seu neurônio e vizinhança;
- `analyze`: amostra que merece ser revisada;
- `remove`: amostra provavelmente ruidosa.

In [ ]:
all_samples <- sits_som_clean_samples(
    som_map             = som_cluster,
    prior_threshold     = 0.6,
    posterior_threshold = 0.6,
    keep                = c("clean", "analyze", "remove")
)

plot(all_samples)

Para manter apenas as amostras `clean` e `analyze`:

In [ ]:
clean_samples <- sits_som_clean_samples(
    som_map = som_cluster,
    keep    = c("clean", "analyze")
)

summary(clean_samples)

<div style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; background-color: #fdfdfd; padding: 15px 20px; margin: 20px 0; font-size: 15px; color: #333; line-height: 1.5; box-shadow: 0 1px 3px rgba(0, 0, 0, 0.05);">
  <b>Nota:</b> neste notebook seguimos o treinamento com todas as amostras (<code>samples_ts</code>). Em um projeto real, as amostras marcadas como <code>remove</code> e <code>analyze</code> devem ser inspecionadas antes de decidir se serão removidas ou corrigidas. Lembre-se também de que <code>clean_samples</code> contém apenas as três bandas usadas no SOM.
</div>

#### Desbalanceamento das amostras
<hr style="border:1px solid #0077b9;">

Como vimos no `summary(samples_ts)`, as classes têm quantidades de amostras muito diferentes: `Soybean` tem mais de 600 amostras, enquanto `Millet` e `Sorghum_Sorghum` têm apenas 10. Esse **desbalanceamento** faz com que o modelo aprenda melhor as classes majoritárias e tenda a ignorar as minoritárias.

O `sits` oferece a função `sits_reduce_imbalance()`, que:

- gera amostras sintéticas para as classes com poucas amostras (*oversampling*, método SMOTE);
- reduz as classes com muitas amostras usando SOM para manter a sua variabilidade (*undersampling*).

<div style="text-align: center">
    <video src="videos/04e-imbalance.mp4" controls preload="metadata" width="85%"></video>
</div>

Não aplicaremos essa etapa neste notebook, mas em casos reais, normalmente é recomendado que seu uso seja feito.

Caso você se encontre no dilema "de usar" ou "não usar" essa operação, recomenda-se que você faça testes e confirme se no seu caso o balanceamento é um problema.

<details style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; padding: 10px 15px; background-color: #fdfdfd; box-shadow: 0 1px 3px rgba(0,0,0,0.05); color: #333;">
  <summary style="font-weight: bold; cursor: pointer; outline: none;"> ⚖️ Exemplo de balanceamento das amostras</summary>

  ```R
  samples_balanced <- sits_reduce_imbalance(
      # Amostras a serem balanceadas
      samples = samples_ts,

      # Número mínimo de amostras por classe (oversampling)
      n_samples_over = 100,

      # Número máximo de amostras por classe (undersampling)
      n_samples_under = 300,

      # Número de processos usados
      multicores = 4
  )

  summary(samples_balanced)
  ```
</details>

<div style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; background-color: #fdfdfd; padding: 15px 20px; margin: 20px 0; font-size: 15px; color: #333; line-height: 1.5; box-shadow: 0 1px 3px rgba(0, 0, 0, 0.05);">
  Para saber mais, consulte os capítulos
  <a href="https://e-sensing.github.io/sitsbook/timeseries.html" target="_blank" style="color: #1f70b8; text-decoration: none;">Working with time series</a>,
  <a href="https://e-sensing.github.io/sitsbook/ts_som.html" target="_blank" style="color: #1f70b8; text-decoration: none;">Self-organized maps for sample quality control</a>
  e
  <a href="https://e-sensing.github.io/sitsbook/ts_balance.html" target="_blank" style="color: #1f70b8; text-decoration: none;">Reducing imbalances in training samples</a>
  do livro oficial do <code>sits</code>.
</div>

## Treinando o modelo de classificação
<hr style="border:1px solid #0077b9;">

Com base nas séries temporais extraídas para cada amostra, é possível treinar um algoritmo de classificação. Nesse processo, o modelo aprende a associar as amostras aos padrões espectro-temporais.

O treinamento no `sits` é feito com a função `sits_train()`. A animação abaixo apresenta a utilização desta função:

<div style="text-align: center">
    <video src="videos/05-sits-train.mp4" controls preload="metadata" width="85%"></video>
</div>

### Treinando modelo para amostras no oeste da Bahia
<hr style="border:1px solid #0077b9;">

Com as amostras extraídas do cubo, faremos o treinamento de um modelo `Random Forest`:

In [ ]:
rfor_model <- sits_train(
  samples   = samples_ts,
  ml_method = sits_rfor()
)

<details style="border: 1px solid #ccc; border-left: 5px solid #1f70b8; border-radius: 8px; padding: 10px 15px; background-color: #fdfdfd; box-shadow: 0 1px 3px rgba(0,0,0,0.05); color: #333;">
  <summary style="font-weight: bold; cursor: pointer; outline: none;"> ⚙️ Mais detalhes sobre os modelos de aprendizado de máquina</summary>
  
  #### Parâmetros adicionais
  Cada modelo tem um conjunto de parâmetros adicionais que podem ser fornecidos. Por exemplo, no método Random Forest, podemos definir o número de árvores que serão geradas usando o parâmetro `num_trees`. Cada modelo tem o seu próprio conjunto de parâmetros.  A documentação dos modelos pode ser verificada utilizando `? + nome_da_função`. O `sits` fornece valores de parâmetros por padrão, mas recomendamos fortemente que consulte a documentação dos modelos.
    
  ```R
  rfor_model <- sits_train(
    # Amostras que serão usadas para o treinamento 
    samples = samples_ts,

    # Parâmetros adicionais do modelo random forest 
    ml_method = sits_rfor(num_trees = 50)
  )
  ```
</details>

### Explorando características do modelo treinado
<hr style="border:1px solid #0077b9;">

In [ ]:
plot(rfor_model)

## Classificação do cubo
<hr style="border:1px solid #0077b9;">

Com o modelo de aprendizado de máquina treinado, podemos gerar um mapa classificado. Para isso, podemos usar a função `sits_classify`, conforme ilustrado na animação abaixo:

<div style="text-align: center">
    <video src="videos/06-sits-classify.mp4" controls preload="metadata" width="85%"></video>
</div>

### Classificação de uma região no oeste da Bahia
<hr style="border:1px solid #0077b9;">

Define uma região de interesse:

In [ ]:
roi <- c(
    lon_min = -46.24898,
    lat_min = -12.68681,
    lon_max = -45.86688,
    lat_max = -12.38479 
)

Define o diretório onde os resultados serão salvos:

In [ ]:
output_dir <- fs::path(Sys.getenv("HOME"))
output_dir <- output_dir / "resultados/classificacao"

fs::dir_create(output_dir, showWarnings = FALSE)

Gera cubo de probabilidades:

In [ ]:
probs <- sits_classify(
    data       = cube,
    ml_model   = rfor_model,
    roi        = roi,
    multicores = 4,
    memsize    = 8,
    output_dir = output_dir,
    version    = "rf"
)

O argumento `version` dá nome aos arquivos gerados. Se você treinar outro modelo, use outra `version` (ou outro `output_dir`). Caso contrário, o `sits` reaproveita os arquivos que já existem (modo de recuperação).

In [ ]:
plot(probs, labels = "Soybean_Corn", palette = "YlOrBr")

## Suavização espacial

<hr style="border:1px solid #0077b9;">

A suavização espacial reduz os ruídos no mapa de probabilidade. No `sits`, a suavização espacial é feita com a função `sits_smooth()`, seguindo o fluxo apresentado na animação abaixo:

<div style="text-align: center">
    <video src="videos/07-sits-smooth.mp4" controls preload="metadata" width="85%"></video>
</div>

### Suavização no mapa de classificação no oeste da Bahia
<hr style="border:1px solid #0077b9;">

Realiza suavização espacial no cubo de probabilidades gerado anteriormente:

In [ ]:
bayes <- sits_smooth(
    cube       = probs,
    multicores = 8,
    memsize    = 16,
    output_dir = output_dir,
    version    = "rf"
)

Visualiza probabilidades suavizadas:

In [ ]:
plot(bayes, labels = "Soybean_Corn", palette = "YlOrBr")

## Associando rótulos ao mapa de classificação

<hr style="border:1px solid #0077b9;">

A associação de rótulos ao mapa de classificação é a etapa final do fluxo principal do `sits`. Nessa fase, são definidas as classes atribuídas a cada *pixel*, como mostrado na animação abaixo:

<div style="text-align: center">
    <video src="videos/08-sits-label.mp4" controls preload="metadata" width="85%"></video>
</div>

### Associando labels ao mapa da Bahia
<hr style="border:1px solid #0077b9;">

Associa labels ao mapa final

In [ ]:
class <- sits_label_classification(
    cube       = bayes,
    multicores = 8,
    memsize    = 16,
    output_dir = output_dir,
    version    = "rf"
)

In [ ]:
plot(class)

In [ ]:
sits_view(class)